# Spending assistant: ask questions about categorised bank-SMS expenses

This notebook is the second half of the prototype. The first notebook (`expense_categoriser_train_test.ipynb`) puts every expense into one of 12 categories. This one lets a person ask questions in plain English, such as *"How much did I spend on food in March 2025?"*, and answers them from the categorised data.

**Design (text-to-SQL, numbers never invented):**
1. The categorised expenses are loaded into a small SQLite database.
2. A small **local** LLM (Qwen2.5-Instruct, run on the Colab GPU) turns the question into one SQL query. No bank data is sent to an external API.
3. A guard accepts only a single read-only `SELECT`, and the database is opened read-only.
4. The answer's numbers come straight from the query result. The LLM never writes a number itself.

**Two evaluations:**
- **Error propagation (no LLM needed):** the same 30 gold SQL questions are answered on the true categories and on categories predicted by the model. This measures how categorisation mistakes turn into wrong spending totals.
- **Text-to-SQL accuracy (needs the LLM):** does the LLM write SQL that returns the gold answer?

Run order: in Colab, run the setup cell, then **Runtime → Run all**. Sections 1–4 run on CPU in about 2 minutes. Section 5 needs a GPU runtime (Runtime → Change runtime type → T4 GPU) and downloads the LLM (about 3 GB for the 1.5B model).

## 0. Setup (Colab + Google Drive)

In [ ]:
# Edit the path if your project folder is elsewhere in Drive.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd "/content/drive/MyDrive/Masters of Applied AI/MAAI902/sms_expense_project_bundle"
except ImportError:
    pass  # running locally: start Jupyter from the project folder

## 1. Config

In [ ]:
USER = 'danish'                 # whose expenses "I" / "my" refers to
TODAY = '2025-12-31'            # data cut-off; "this month" = 2025-12, "last month" = 2025-11
LLM_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'   # step up: 'Qwen/Qwen2.5-3B-Instruct' (fits a T4 in fp16)
RUN_LLM = True                  # False = skip section 5 (e.g. no GPU)
RANDOM_STATE = 42

## 2. Load expenses and categories, and make honest predicted categories

For the error-propagation test the predicted categories must come from a model that **did not see** those transactions. We use out-of-fold predictions from the same TF-IDF + logistic regression model as the first notebook, in two settings:
- **known merchants** (random 5-fold): the everyday case, where most merchants have been seen before;
- **unseen merchants** (grouped 5-fold): the worst case, where every merchant is new to the model.

In [ ]:
import sys, re, sqlite3, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GroupKFold
warnings.filterwarnings('ignore')
sys.path.insert(0, 'scripts')
from categoriser_lib import load_labelled, tfidf_logreg, FEATURES
from rules import CATEGORIES

personal, khalifa, CATS = load_labelled()

def out_of_fold(df, splitter, groups=None):
    pred = pd.Series(index=df.index, dtype=object)
    splits = splitter.split(df, df.label) if groups is None else splitter.split(df, groups=groups)
    for tr, te in splits:
        m = tfidf_logreg().fit(df.iloc[tr][FEATURES], df.iloc[tr].label)
        pred.iloc[te] = m.predict(df.iloc[te][FEATURES])
    return pred

personal['pred_known'] = out_of_fold(personal, StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE))
personal['pred_unseen'] = out_of_fold(personal, GroupKFold(5), groups=personal.merchant_clean)
for c in ['pred_known', 'pred_unseen']:
    print(f'{c}: transaction accuracy {(personal[c] == personal.label).mean():.1%}')

In [ ]:
def build_db(category_col):
    # In-memory SQLite DB with one table, `expenses`, using the chosen category column.
    t = personal.assign(
        date=personal.date.dt.strftime('%Y-%m-%d'),
        month=personal.date.dt.strftime('%Y-%m'),
        year=personal.date.dt.year,
        category=personal[category_col],
        amount_aed=personal.amount_aed.round(2),
    )[['txn_id', 'user', 'bank', 'date', 'month', 'year', 'merchant_clean', 'amount_aed', 'category']]
    t = t.rename(columns={'merchant_clean': 'merchant'})
    con = sqlite3.connect(':memory:')
    t.to_sql('expenses', con, index=False)
    return con

DBS = {'true': build_db('label'), 'pred_known': build_db('pred_known'), 'pred_unseen': build_db('pred_unseen')}
pd.read_sql('SELECT * FROM expenses LIMIT 5', DBS['true'])

## 3. Gold question set

30 questions a user might ask, each with a hand-written gold SQL query. `uses_category` marks questions whose answer depends on categories (20) versus merchant names, dates or totals only (10). The second group is a control: categorisation errors cannot change those answers.

In [ ]:
U = f"user = '{USER}'"
Q = [  # (question, gold SQL, uses_category)
 ("How much did I spend on food and dining in May 2025?",
  f"SELECT ROUND(SUM(amount_aed),2) FROM expenses WHERE {U} AND category='Food_Dining' AND month='2025-05'", True),
 ("How much did I spend on groceries in 2025?",
  f"SELECT ROUND(SUM(amount_aed),2) FROM expenses WHERE {U} AND category='Groceries' AND year=2025", True),
 ("What did I spend on transport in July 2025?",
  f"SELECT ROUND(SUM(amount_aed),2) FROM expenses WHERE {U} AND category='Transport_Auto' AND month='2025-07'", True),
 ("How much went on digital subscriptions in 2025?",
  f"SELECT ROUND(SUM(amount_aed),2) FROM expenses WHERE {U} AND category='Digital_Subscriptions' AND year=2025", True),
 ("How much did I spend on entertainment in 2024?",
  f"SELECT ROUND(SUM(amount_aed),2) FROM expenses WHERE {U} AND category='Entertainment' AND year=2024", True),
 ("What was my total shopping spend from July to December 2025?",
  f"SELECT ROUND(SUM(amount_aed),2) FROM expenses WHERE {U} AND category='Shopping_Retail' AND month BETWEEN '2025-07' AND '2025-12'", True),
 ("How much did I spend on health and wellness in 2025?",
  f"SELECT ROUND(SUM(amount_aed),2) FROM expenses WHERE {U} AND category='Health_Wellness' AND year=2025", True),
 ("How much were my utility and telecom bills in 2025?",
  f"SELECT ROUND(SUM(amount_aed),2) FROM expenses WHERE {U} AND category='Utilities_Telecom' AND year=2025", True),
 ("Compare my food and dining spending in October and December 2025.",
  f"SELECT month, ROUND(SUM(amount_aed),2) FROM expenses WHERE {U} AND category='Food_Dining' AND month IN ('2025-10','2025-12') GROUP BY month ORDER BY month", True),
 ("Which category did I spend the most on in 2025?",
  f"SELECT category FROM expenses WHERE {U} AND year=2025 GROUP BY category ORDER BY SUM(amount_aed) DESC LIMIT 1", True),
 ("What were my top 3 spending categories in 2024?",
  f"SELECT category FROM expenses WHERE {U} AND year=2024 GROUP BY category ORDER BY SUM(amount_aed) DESC LIMIT 3", True),
 ("Which month in 2025 had my highest transport spending?",
  f"SELECT month FROM expenses WHERE {U} AND category='Transport_Auto' AND year=2025 GROUP BY month ORDER BY SUM(amount_aed) DESC LIMIT 1", True),
 ("What percentage of my 2025 spending went on food and dining?",
  f"SELECT ROUND(100.0*SUM(CASE WHEN category='Food_Dining' THEN amount_aed ELSE 0 END)/SUM(amount_aed),1) FROM expenses WHERE {U} AND year=2025", True),
 ("What was my average monthly grocery spend in 2025?",
  f"SELECT ROUND(SUM(amount_aed)/12.0,2) FROM expenses WHERE {U} AND category='Groceries' AND year=2025", True),
 ("How many food and dining purchases did I make this month?",
  f"SELECT COUNT(*) FROM expenses WHERE {U} AND category='Food_Dining' AND month='2025-12'", True),
 ("How much did I spend on travel in 2025?",
  f"SELECT ROUND(SUM(amount_aed),2) FROM expenses WHERE {U} AND category='Travel' AND year=2025", True),
 ("How much did I pay in government fees in 2025?",
  f"SELECT ROUND(SUM(amount_aed),2) FROM expenses WHERE {U} AND category='Government_Fees' AND year=2025", True),
 ("What was my biggest single entertainment purchase in 2025?",
  f"SELECT ROUND(MAX(amount_aed),2) FROM expenses WHERE {U} AND category='Entertainment' AND year=2025", True),
 ("How much did I spend on entertainment last month?",
  f"SELECT ROUND(SUM(amount_aed),2) FROM expenses WHERE {U} AND category='Entertainment' AND month='2025-11'", True),
 ("How much did I spend on transport in 2024 compared with 2025?",
  f"SELECT year, ROUND(SUM(amount_aed),2) FROM expenses WHERE {U} AND category='Transport_Auto' AND year IN (2024,2025) GROUP BY year ORDER BY year", True),
 # controls: no category involved
 ("How much did I spend at Amazon in 2025?",
  f"SELECT ROUND(SUM(amount_aed),2) FROM expenses WHERE {U} AND merchant='Amazon' AND year=2025", False),
 ("How many times did I pay Snapchat in 2025?",
  f"SELECT COUNT(*) FROM expenses WHERE {U} AND merchant='Snapchat' AND year=2025", False),
 ("How much did I spend at McDonald's in 2025?",
  f"SELECT ROUND(SUM(amount_aed),2) FROM expenses WHERE {U} AND merchant='McDonald''s' AND year=2025", False),
 ("How much did I spend in total this month?",
  f"SELECT ROUND(SUM(amount_aed),2) FROM expenses WHERE {U} AND month='2025-12'", False),
 ("How much did I spend in total last month?",
  f"SELECT ROUND(SUM(amount_aed),2) FROM expenses WHERE {U} AND month='2025-11'", False),
 ("Compare my total spending in 2024 and 2025.",
  f"SELECT year, ROUND(SUM(amount_aed),2) FROM expenses WHERE {U} AND year IN (2024,2025) GROUP BY year ORDER BY year", False),
 ("Which merchant did I spend the most at in 2025?",
  f"SELECT merchant FROM expenses WHERE {U} AND year=2025 GROUP BY merchant ORDER BY SUM(amount_aed) DESC LIMIT 1", False),
 ("What was my average purchase amount in 2025?",
  f"SELECT ROUND(AVG(amount_aed),2) FROM expenses WHERE {U} AND year=2025", False),
 ("How much did I spend on Bolt rides in 2025?",
  f"SELECT ROUND(SUM(amount_aed),2) FROM expenses WHERE {U} AND merchant='Bolt' AND year=2025", False),
 ("Which month in 2025 did I spend the most overall?",
  f"SELECT month FROM expenses WHERE {U} AND year=2025 GROUP BY month ORDER BY SUM(amount_aed) DESC LIMIT 1", False),
]
QS = pd.DataFrame(Q, columns=['question', 'gold_sql', 'uses_category'])

def run(sql, con):
    return [tuple(r) for r in con.execute(sql).fetchall()]

QS['gold_answer'] = [run(s, DBS['true']) for s in QS.gold_sql]
pd.set_option('display.max_colwidth', 80)
QS[['question', 'uses_category', 'gold_answer']]

## 4. Error propagation: do categorisation mistakes change the answers?

Each gold query is run on the true categories and on the predicted categories. A numeric answer counts as **correct** if it is within 1% (or 1 AED) of the true answer; a text answer (a category, month or merchant) must match exactly. For numeric answers we also report the absolute percentage error.

In [ ]:
def same(a, b, tol=0.01):
    if len(a) != len(b):
        return False
    for ra, rb in zip(a, b):
        for x, y in zip(ra, rb):
            if isinstance(x, (int, float)) or isinstance(y, (int, float)):
                x, y = float(x or 0), float(y or 0)
                if abs(x - y) > max(1.0, tol * abs(x)):
                    return False
            elif x != y:
                return False
    return True

def pct_error(a, b):
    xs = [float(v or 0) for r in a for v in r if isinstance(v, (int, float)) or v is None]
    ys = [float(v or 0) for r in b for v in r if isinstance(v, (int, float)) or v is None]
    if not xs or len(xs) != len(ys) or any(isinstance(v, str) for r in a for v in r):
        return np.nan
    return np.mean([abs(x - y) / max(abs(x), 1.0) for x, y in zip(xs, ys)])

for k in ['pred_known', 'pred_unseen']:
    QS[f'ans_{k}'] = [run(s, DBS[k]) for s in QS.gold_sql]
    QS[f'ok_{k}'] = [same(g, a) for g, a in zip(QS.gold_answer, QS[f'ans_{k}'])]
    QS[f'err_{k}'] = [pct_error(g, a) for g, a in zip(QS.gold_answer, QS[f'ans_{k}'])]

prop = QS.groupby('uses_category').agg(
    questions=('question', 'size'),
    correct_known=('ok_pred_known', 'mean'), median_err_known=('err_pred_known', 'median'),
    correct_unseen=('ok_pred_unseen', 'mean'), median_err_unseen=('err_pred_unseen', 'median'))
prop.index = prop.index.map({True: 'category questions', False: 'control questions (no category)'})
prop.style.format({c: '{:.0%}' for c in prop.columns if c != 'questions'})

In [ ]:
# Where the wrong answers come from
show = QS[QS.uses_category][['question', 'gold_answer', 'ans_pred_known', 'ans_pred_unseen', 'err_pred_known', 'err_pred_unseen']]
show.style.format({'err_pred_known': '{:.0%}', 'err_pred_unseen': '{:.0%}'}, na_rep='-')

## 5. Text-to-SQL with a local LLM (GPU)

The LLM gets the table schema, the 12 category names with one-line descriptions, the date conventions, and two worked examples that are **not** in the gold set. It must answer with one SQL query only.

**Guard rails** (the LLM's SQL is never trusted blindly):
- only one statement, and it must start with `SELECT` or `WITH`;
- keywords that change data or the database (`INSERT`, `UPDATE`, `DELETE`, `DROP`, `ALTER`, `ATTACH`, `PRAGMA`, `CREATE`, `REPLACE`) are rejected;
- the query runs on a read-only copy of the data, with a time limit.

In [ ]:
SCHEMA = '''Table expenses (one row per card purchase or bill payment):
  txn_id TEXT, user TEXT, bank TEXT, date TEXT 'YYYY-MM-DD', month TEXT 'YYYY-MM', year INTEGER,
  merchant TEXT (cleaned brand name, e.g. 'Amazon', 'McDonald''s', 'Snapchat', 'Bolt', 'ENOC'),
  amount_aed REAL (positive, in AED), category TEXT'''
CAT_TEXT = '\n'.join(f"  {c}: {CATEGORIES[c]}" for c in CATS)
SYSTEM = f'''You write one SQLite query that answers a question about the user's spending.
{SCHEMA}
Allowed category values:
{CAT_TEXT}
Rules: always filter user = '{USER}'. Today is {TODAY}: "this month" = month '2025-12', "last month" = '2025-11'.
Round money with ROUND(x,2). Return only the SQL, no explanation, no markdown.'''
FEWSHOT = [
  ("How much did I spend on groceries in June 2025?",
   f"SELECT ROUND(SUM(amount_aed),2) FROM expenses WHERE user = '{USER}' AND category='Groceries' AND month='2025-06'"),
  ("Which merchant did I pay most often in 2024?",
   f"SELECT merchant FROM expenses WHERE user = '{USER}' AND year=2024 GROUP BY merchant ORDER BY COUNT(*) DESC LIMIT 1"),
]

BANNED = re.compile(r'\b(INSERT|UPDATE|DELETE|DROP|ALTER|ATTACH|DETACH|PRAGMA|CREATE|REPLACE|VACUUM)\b', re.I)
def guard(sql):
    sql = sql.strip().strip('`').strip()
    sql = re.sub(r'^sql\s*', '', sql, flags=re.I).strip().rstrip(';').strip()
    if ';' in sql:
        raise ValueError('more than one statement')
    if not re.match(r'^(SELECT|WITH)\b', sql, re.I):
        raise ValueError('not a SELECT query')
    if BANNED.search(sql):
        raise ValueError('forbidden keyword')
    return sql

def readonly_copy(con):
    # Fresh in-memory copy opened read-only, so even a bad query cannot change the data.
    import tempfile, os
    path = os.path.join(tempfile.mkdtemp(), 'expenses.db')
    disk = sqlite3.connect(path); con.backup(disk); disk.close()
    ro = sqlite3.connect(f'file:{path}?mode=ro', uri=True)
    return ro

def safe_run(sql, con, max_steps=5_000_000):
    steps = [0]
    def tick():
        steps[0] += 1
        return 1 if steps[0] > max_steps / 1000 else 0
    con.set_progress_handler(tick, 1000)
    try:
        return [tuple(r) for r in con.execute(sql).fetchall()]
    finally:
        con.set_progress_handler(None, 0)

RO = readonly_copy(DBS['true'])

In [ ]:
llm = None
if RUN_LLM:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    if not torch.cuda.is_available():
        print('No GPU found: set Runtime -> Change runtime type -> T4 GPU, or RUN_LLM = False.')
    else:
        tok = AutoTokenizer.from_pretrained(LLM_MODEL)
        llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL, dtype=torch.float16, device_map='cuda')
        print('Loaded', LLM_MODEL)

def to_sql(question):
    msgs = [{'role': 'system', 'content': SYSTEM}]
    for q, s in FEWSHOT:
        msgs += [{'role': 'user', 'content': q}, {'role': 'assistant', 'content': s}]
    msgs.append({'role': 'user', 'content': question})
    # Build the prompt as text, then tokenise it (works on transformers v4 and v5)
    prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    inputs = tok(prompt, return_tensors='pt').to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=200, do_sample=False,   # greedy = reproducible
                           pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
if llm is not None:
    rows = []
    for q, gold in zip(QS.question, QS.gold_answer):
        raw = to_sql(q)
        try:
            sql = guard(raw); ans = safe_run(sql, RO); ok = same(gold, ans); err = ''
        except Exception as e:
            sql, ans, ok, err = raw, None, False, str(e)[:60]
        rows.append(dict(question=q, llm_sql=sql, llm_answer=ans, correct=ok, error=err))
    LLM_RES = pd.DataFrame(rows)
    QS['llm_correct'] = LLM_RES.correct.values
    print(f'Execution accuracy ({LLM_MODEL}): {LLM_RES.correct.mean():.0%} '
          f'| category questions {QS[QS.uses_category].llm_correct.mean():.0%} '
          f'| control questions {QS[~QS.uses_category].llm_correct.mean():.0%} '
          f'| rejected or failed queries {(LLM_RES.error != "").sum()}')
    display(LLM_RES)
else:
    print('Skipped: LLM not loaded.')

## 6. Ask your own questions

`ask()` runs the whole pipeline: question → SQL (LLM) → guard → read-only query → answer. Numbers in the reply are copied from the query result; the LLM never writes them. Advice questions (*"How can I save 1,000 AED?"*) get a summary of the real category totals to reason from, plus a reminder that this is not financial advice.

In [ ]:
ADVICE = re.compile(r'\b(save|saving|cut|reduce|budget|advice|tips?)\b', re.I)

def category_summary(months=3):
    last = pd.read_sql(f"SELECT DISTINCT month FROM expenses WHERE user='{USER}' ORDER BY month DESC LIMIT {months}", RO).month
    q = (f"SELECT category, ROUND(SUM(amount_aed)/{months}.0,0) AS avg_per_month FROM expenses "
         f"WHERE user='{USER}' AND month IN ({','.join(repr(m) for m in last)}) GROUP BY category ORDER BY 2 DESC")
    return pd.read_sql(q, RO)

def ask(question):
    if llm is None:
        return 'LLM not loaded (needs a GPU runtime).'
    if ADVICE.search(question):
        table = category_summary()
        prompt = [{'role': 'system', 'content': 'You are a careful budgeting helper. Use ONLY the numbers given. '
                   'Suggest 3 specific, realistic cuts and how much each could save per month. '
                   'Do not invent numbers. End with: "This is not financial advice."'},
                  {'role': 'user', 'content': f'Average monthly spending by category (last 3 months, AED):\n{table.to_string(index=False)}\n\nQuestion: {question}'}]
        # Build the prompt as text, then tokenise it (works on transformers v4 and v5)
        text = tok.apply_chat_template(prompt, add_generation_prompt=True, tokenize=False)
        inputs = tok(text, return_tensors='pt').to(llm.device)
        with torch.no_grad():
            out = llm.generate(**inputs, max_new_tokens=300, do_sample=False,
                               pad_token_id=tok.eos_token_id)
        return tok.decode(out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    try:
        sql = guard(to_sql(question))
        rows = safe_run(sql, RO)
    except Exception as e:
        return f'Sorry, I could not answer that safely ({e}).'
    if not rows or rows == [(None,)]:
        return f'No matching expenses found.\n\nSQL used:\n{sql}'
    value = '; '.join(', '.join(f'{v:,.2f}' if isinstance(v, float) else str(v) for v in r) for r in rows)
    return f'{value}\n\nSQL used:\n{sql}'

for q in ['How much did I spend on food and dining in December 2025?',
          'What were my top 3 spending categories in 2025?',
          'How can I save 1,000 AED a month?']:
    print('Q:', q); print('A:', ask(q)); print('-' * 60)

In [ ]:
# Optional chat window inside Colab (no public link: share=False).
# import gradio as gr
# gr.Interface(fn=ask, inputs=gr.Textbox(label='Ask about your spending'),
#              outputs=gr.Textbox(label='Answer'), title='UAE bank-SMS spending assistant').launch(share=False)

## Notes for the write-up
- **Error propagation** is the bridge between categorisation accuracy and user-facing value: a question about Amazon or a monthly total is unaffected by categorisation, but a question about "food" inherits every categorisation error. Report the share of category questions answered correctly with known-merchant and unseen-merchant predictions, next to the 100% for control questions.
- The LLM only writes SQL. Every number shown to the user comes from the database, which removes the main hallucination risk for factual questions. Advice answers are the exception: they are free text, grounded in real totals and labelled as not financial advice.
- The 30 questions are a small, hand-written set over one user; treat the results as indicative.